# Exploratory Data Analysis (EDA)
## Antalya/Alanya Resort Hotels: Occupancy Prediction with Google Trends

**Hazırlayan:** Bedirhan Sarı  
**Ders:** DSA210 - Introduction to Data Science  
**Dönem:** Spring 2025-2026  
**Kurum:** Sabancı University

---

## Proje Özeti

Bu projenin amacı, Antalya/Alanya bölgesindeki resort oteller için **günlük doluluk oranını (occupancy rate)** tahmin etmeye yardımcı olabilecek sinyalleri incelemektir. Projenin çıkış noktası, bu otellerin sadece bireysel müşterilerle değil, yoğun şekilde **B2B kanal** üzerinden, yani tur operatörleri ve acenteler aracılığıyla çalışmasıdır.

Bu yüzden temel araştırma sorusu şudur:

**Google Trends gibi dijital talep göstergeleri, bu tip resort otellerin doluluk oranını açıklamada veya tahmin etmede gerçekten işe yarıyor mu?**

Bu soruya cevap vermek için iki farklı veri kaynağı bir araya getirildi:

1. **Otel doluluk verileri**  
   - Side Mare Hotel  
   - Azura Deluxe  
2. **Google Trends verileri**  
   - Germany  
   - Netherlands  
   - United Kingdom  
   - Türkiye  

Bu notebook, projenin **14 Nisan teslimine** karşılık gelen kısmını, yani:
- veri toplama,
- veri birleştirme,
- keşifsel veri analizi (EDA),
- hipotez testi niteliğindeki ilk istatistiksel incelemeleri

detaylı biçimde açıklamak için hazırlanmıştır.


## Problem Tanımı ve Motivasyon

Turizm sektörü, özellikle resort oteller için, çok güçlü mevsimsellik taşıyan ve dışsal etkenlere duyarlı bir alandır. Antalya/Alanya hattında faaliyet gösteren otellerde doluluk oranı aşağıdaki faktörlerden etkilenebilir:

- yaz sezonu / düşük sezon,
- ülke bazlı tatil dönemleri,
- kur hareketleri,
- dijital arama ilgisi,
- tur operatörlerinin yönlendirdiği talep.

Bu projede odak noktamız şudur:

> Eğer insanlar belirli ülkelerde Antalya, Alanya veya benzeri tatil anahtar kelimelerini daha fazla arıyorsa, bu artış otellerin doluluk oranında daha sonra görülebilir mi?

Buradaki kritik nokta, ilişkinin **aynı gün** ortaya çıkmasının beklenmemesidir. İnsanlar bir destinasyonu aradıktan sonra:
- karar verir,
- rezervasyon yapar,
- seyahati planlar,
- daha sonra otele gelir.

Bu yüzden proje sadece aynı gün ilişkisine değil, **gecikmeli (lagged)** ilişkilere de bakmaktadır.


## Veri Kaynakları

### 1. Otel Verileri
İki otel için günlük doluluk oranı verileri hazırlandı:
- **Side Mare Hotel**
- **Azura Deluxe**

Bu dosyalarda temel hedef değişken:
- `occupancy_rate`

Her satır, belirli bir tarihte belirli bir otelin doluluk oranını temsil etmektedir.

### 2. Google Trends Verileri
Google Trends tarafında dört ülke için turizmle ilgili anahtar kelimeler toplandı:
- Germany
- Netherlands
- United Kingdom
- Türkiye

Google Trends veri yapısı başlangıçta şu mantıktaydı:
- tarih,
- ülke,
- anahtar kelime,
- trend skoru

Bu skorlar daha sonra analiz ve modelleme için uygun olacak şekilde düzenlendi.

---

## Neden Veri Zenginleştirme Yapıldı?

Tek başına otel doluluk verisi, bize sadece **sonucu** gösterir.  
Ancak doluluğun neden arttığını veya azaldığını anlamak için dışsal değişkenlere ihtiyaç vardır.

Google Trends burada bir **talep sinyali** olarak düşünülmüştür:
- potansiyel turist ilgisi,
- destinasyon araştırma davranışı,
- tatil planlama niyeti.

Bu nedenle Trends verisi, otel verisine eklenmiş ve proje veri zenginleştirme şartını karşılayacak şekilde kullanılmıştır.


In [ ]:
import pandas as pd

side = pd.read_excel("side_mare_daily_occupancy_combined.xlsx")
azura = pd.read_excel("azura_deluxe_daily_occupancy_combined.xlsx")

tr_de = pd.read_excel("combined_trends_germany.xlsx")
tr_nl = pd.read_excel("combined_trends_netherlands.xlsx")
tr_uk = pd.read_excel("combined_trends_united_kingdom.xlsx")
tr_tr = pd.read_excel("combined_trends_turkiye.xlsx")


## Veri Hazırlama Süreci

Bu çalışmada ilk önemli teknik adım, birbirinden bağımsız gelen dosyaları tek bir analiz yapısına dönüştürmek oldu.

### Yapılan ana işlemler

#### A. Otel dosyalarının standardizasyonu
İki otel dosyasında ortak sütun yapısı oluşturuldu:
- `date`
- `hotel_name`
- `occupancy_rate`

Böylece iki otel verisi aynı formatta tek tabloda birleştirilebildi.

#### B. Trends dosyalarının standardizasyonu
Google Trends dosyalarında sütun isimleri ve veri tipleri düzenlendi:
- `date`
- `country`
- `keyword`
- `google_trends_score`

#### C. Long formattan wide formata geçiş
Google Trends verisi başlangıçta her tarih için birden fazla satır içeriyordu. Çünkü aynı tarihte:
- farklı ülke,
- farklı anahtar kelime,
- farklı trend skoru

vardı.

Bunu analiz edebilmek için veri pivot edilerek şu tip sütunlar oluşturuldu:
- `trends_germany_alanya_hotel`
- `trends_turkiye_side_otel`
- `trends_netherlands_hotel_alanya`
- vb.

Bu sayede her tarih için tüm Trends değişkenleri yan yana geldi.

#### D. Master table oluşturulması
Son aşamada otel verileri ile Trends verileri `date` üzerinden birleştirildi.

Böylece oluşan ana tablo şu mantığa sahip oldu:

**bir satır = bir tarih + bir otel**


## Master Table Nedir?

Master table, projedeki tüm analizlerin temelini oluşturan birleşik tablodur.

İçeriğinde:
- hedef değişken olarak `occupancy_rate`,
- açıklayıcı değişkenler olarak Trends sütunları,
- her otel için tarih bazlı kayıtlar

yer alır.

Bu tablo olmadan:
- korelasyon analizi,
- gecikmeli ilişki incelemesi,
- modelleme

çok dağınık hale gelirdi.

Bu yüzden proje akışındaki en kritik hazırlık adımı, master table'ın kurulmasıdır.


In [ ]:
master = pd.read_excel("hotel_master_table.xlsx", sheet_name="master_table")
master["date"] = pd.to_datetime(master["date"])

master.head()


## Veri Kalitesi Kontrolleri

EDA'ya geçmeden önce veri setinin güvenilir olup olmadığını kontrol etmek gerekir. Bu nedenle aşağıdaki kontroller yapıldı:

### 1. Eksik veri kontrolü
Özellikle Trends sütunlarında eksik değer olup olmadığı incelendi.

### 2. Kopya satır kontrolü
Aynı `date` ve `hotel_name` kombinasyonunun birden fazla kez geçip geçmediği test edildi.

### 3. Tarih aralığı kontrolü
Otel verileri ile Trends verilerinin tarih aralıklarının ne kadar örtüştüğü incelendi.

### 4. Veri tipi kontrolü
- tarih sütunu datetime mı?
- occupancy_rate sayısal mı?
- Trends skorları sayısal mı?

Bu kontroller, daha sonra yapılacak istatistiksel analizlerin ve makine öğrenmesi adımlarının sağlam olmasını sağlar.


In [ ]:
master.info()
master.isna().sum().sort_values(ascending=False).head(20)


## İlk Gözlemler

İlk kalite kontrollerinden çıkan önemli noktalar şunlardı:

- Veri seti genel olarak analiz için uygundu.
- Aynı tarih ve otel için tekrar eden satır bulunmadı.
- Bazı Trends sütunlarında eksik değerler vardı.
- Bu eksik değerlerin temel nedeni, Trends veri aralığı ile otel veri aralığının başlangıçta tam örtüşmemesiydi.

Bu durum önemli ama kritik bir hata değil. Çünkü:
- eksikler sistematik biçimde anlaşılabiliyor,
- hangi dönemlerden kaynaklandığı biliniyor,
- ilerleyen adımlarda buna uygun işlem yapılabiliyor.


## Araştırma Hipotezleri

Bu EDA aşamasında test edilmek istenen temel hipotezler şunlardır:

### H1
Google Trends değişkenleri ile otel doluluk oranı arasında ilişki vardır.

### H2
Google Trends'in aynı gün değeri yerine, birkaç gün veya birkaç hafta önceki değeri daha güçlü ilişki gösterir.

### H3
Her ülke ve anahtar kelime aynı derecede yararlı değildir; bazı ülke-kelime kombinasyonları daha güçlü sinyal taşır.

### H4
Doluluk oranı çok güçlü şekilde mevsimsellik taşır; bu yüzden herhangi bir dışsal değişken değerlendirilirken sezon etkisi göz önünde tutulmalıdır.


## Univariate EDA: Doluluk Oranını Tek Başına İncelemek

İlk olarak hedef değişken olan `occupancy_rate` tek başına incelendi.

Bu aşamada bakılan başlıca sorular:
- Doluluk oranı genel olarak hangi aralıkta yoğunlaşıyor?
- İki otel arasında ortalama doluluk farkı var mı?
- Çok düşük veya çok yüksek değerler hangi dönemlerde oluşuyor?
- Mevsimsellik açıkça gözüküyor mu?

Bu analiz sayesinde hedef değişkenin davranışı anlaşılmadan başka bir değişkenle ilişki kurmaya çalışılmamış oldu.


In [ ]:
master.groupby("hotel_name")["occupancy_rate"].agg(["count", "mean", "median", "std", "min", "max"])


## Doluluk Açısından İlk Yorumlar

Yapılan ilk özet istatistikler, iki otelin yapısal olarak benzer ama tamamen aynı olmayan davranış sergilediğini gösterdi.

Öne çıkan noktalar:
- Side Mare Hotel'in ortalama doluluk seviyesi daha yüksekti.
- Azura Deluxe tarafında değişkenlik biraz daha fazlaydı.
- Her iki otelde de sezon etkisi çok belirgindi.
- Bazı dönemlerde doluluk sıfıra çok yaklaşan veya sıfır olan gözlemler görüldü. Bunlar sezon kapanışı, düşük sezon veya operasyonel nedenlerle ilişkili olabilir.

Bu bulgular, daha en baştan şu mesajı veriyor:
**Bu veride güçlü bir takvim ve sezon etkisi var.**


## Zaman İçinde Doluluk Davranışı

Sadece özet istatistikler yeterli değildir. Çünkü zaman serilerinde asıl anlamlı bilgi, değişkenin zaman içindeki akışında ortaya çıkar.

Bu yüzden:
- günlük doluluk oranı zaman ekseninde çizildi,
- oteller ayrı ayrı gözlemlendi,
- aylık ortalamalara da bakılarak daha yumuşak bir sezon resmi çıkarıldı.

Bu görsellerin amacı:
- yaz sezonu ve düşük sezon geçişlerini görmek,
- otellerin benzer hareket edip etmediğini anlamak,
- ani kırılmaları fark etmekti.


## Google Trends Değişkenlerini İncelemek

Doluluk oranı incelendikten sonra sıra Trends değişkenlerine geldi.

Buradaki amaç:
- hangi anahtar kelimeler sürekli sıfıra yakın?
- hangilerinde sezonsal hareket var?
- hangileri çok gürültülü?
- hangileri görsel olarak dolulukla benzer hareket ediyor olabilir?

Bu aşamada her Trends sütununa sadece model girdisi gibi bakılmadı; önce kendi başına nasıl davrandığı anlaşılmaya çalışıldı.

Bu çok önemlidir çünkü bazı özellikler teoride mantıklı görünse bile pratikte neredeyse sabit veya çok gürültülü olabilir.


## Bivariate EDA: Trends ile Doluluğu Birlikte İncelemek

Asıl araştırma sorusuna yaklaşmak için, şimdi iki değişkenin birlikte hareketine bakıldı:

- `occupancy_rate`
- her bir Google Trends özelliği

İlk bakılan şey **aynı gün ilişkisi** oldu.

Yani şu soru soruldu:

> Bugünün Trends skoru ile bugünün doluluk oranı arasında ilişki var mı?

Bunun için iki temel korelasyon ölçüsü kullanıldı:

### Pearson korelasyonu
Doğrusal ilişkiyi ölçer.

### Spearman korelasyonu
Sıralama temelli ilişkiyi ölçer; doğrusal olmasa bile monoton bir yapı varsa onu da yakalayabilir.


In [ ]:
from scipy.stats import pearsonr, spearmanr

trend_cols = [c for c in master.columns if c.startswith("trends_")]

rows = []
for c in trend_cols:
    sub = master[["occupancy_rate", c]].dropna()
    if len(sub) > 10 and sub[c].nunique() > 1:
        p = pearsonr(sub["occupancy_rate"], sub[c])
        s = spearmanr(sub["occupancy_rate"], sub[c], nan_policy="omit")
        rows.append({
            "feature": c,
            "pearson_r": p.statistic,
            "pearson_p": p.pvalue,
            "spearman_rho": s.statistic,
            "spearman_p": s.pvalue,
        })

same_day_corr = pd.DataFrame(rows).sort_values("pearson_r", ascending=False)
same_day_corr.head(10)


## Aynı Gün Korelasyonlarının Yorumu

İlk sonuçlar şunu gösterdi:

- Aynı gün ilişkileri genel olarak **zayıf ile orta** düzey arasındaydı.
- Yani Google Trends, doluluk oranını aynı gün açıklayan çok güçlü bir değişken gibi davranmıyordu.
- Bu durum proje motivasyonuyla uyumlu çıktı.

Çünkü başta da beklenen şey şuydu:
Bu resort otellerde doluluk yalnızca anlık bireysel arama ilgisine bağlı değildir.  
B2B yapı, önceden planlanmış rezervasyonlar ve operatör etkisi oldukça güçlü olabilir.

Dolayısıyla aynı gün korelasyonunun sınırlı olması, proje fikrini çürütmek yerine aslında daha gerçekçi bir çerçeve sunar.


## Gecikmeli İlişki (Lag) Neden Önemli?

Turizmde arama davranışı ile konaklama arasında doğal bir zaman farkı vardır.

Bir kişi:
1. bugün Antalya araması yapabilir,
2. birkaç gün sonra rezervasyon verebilir,
3. birkaç hafta sonra seyahat edebilir.

Bu yüzden sadece aynı gün ilişkisine bakmak eksik olur.

Bunu hesaba katmak için şu gecikmeler test edildi:
- 7 gün
- 14 gün
- 21 gün
- 28 gün

Buradaki mantık:
örneğin 28 günlük lag testinde, bugünkü doluluk ile **28 gün önceki** Trends değeri karşılaştırılır.


In [ ]:
trend_df = master[["date"] + trend_cols].drop_duplicates("date").sort_values("date").reset_index(drop=True)

lag_rows = []
for lag in [7, 14, 21, 28]:
    lagged = trend_df.copy()
    lagged[trend_cols] = lagged[trend_cols].shift(lag)
    merged = master[["date", "hotel_name", "occupancy_rate"]].merge(lagged, on="date", how="left")
    for c in trend_cols:
        sub = merged[["occupancy_rate", c]].dropna()
        if len(sub) > 10 and sub[c].nunique() > 1:
            p = pearsonr(sub["occupancy_rate"], sub[c])
            lag_rows.append({
                "feature": c,
                "lag_days": lag,
                "pearson_r": p.statistic,
                "pearson_p": p.pvalue,
            })

lag_corr = pd.DataFrame(lag_rows)
best_by_feature = lag_corr.loc[lag_corr.groupby("feature")["pearson_r"].idxmax()].sort_values("pearson_r", ascending=False)
best_by_feature.head(10)


## Lag Sonuçlarının Yorumu

Bu aşama, projenin en önemli erken bulgularını verdi.

Sonuçlar gösterdi ki:
- bazı Trends özellikleri, aynı gün versiyonlarına göre gecikmeli kullanıldığında daha güçlü ilişki veriyor,
- yani arama ilgisi ile doluluk arasında gerçekten bir zaman farkı olabilir.

En güçlü örneklerden biri:
- `trends_turkiye_side_otel` değişkeninin **28 günlük lag** ile daha yüksek ilişki göstermesiydi.

Bu bulgu iki açıdan önemli:

### 1. Teorik açıdan
Arama -> planlama -> rezervasyon -> konaklama zinciri mantıklı görünüyor.

### 2. Pratik açıdan
Makine öğrenmesi aşamasında Trends değişkenlerini direkt değil, **lag'li özellikler** olarak kullanmak daha anlamlı olabilir.


## Hipotezlerin Değerlendirilmesi

### H1: Google Trends ile doluluk arasında ilişki vardır.
**Kısmen desteklendi.**  
Tüm değişkenler için güçlü ilişki bulunmadı; ancak bazı ülke-kelime kombinasyonlarında anlamlı ve kullanılabilir ilişkiler görüldü.

### H2: Gecikmeli Trends özellikleri aynı gün özelliklerinden daha faydalıdır.
**Desteklendi.**  
Birçok değişkende lag kullanımı korelasyonu yükseltti.

### H3: Ülke ve anahtar kelimeye göre ilişki gücü değişir.
**Desteklendi.**  
Bazı Türkiye ve Almanya bazlı aramalar daha güçlü görünürken, bazı UK anahtar kelimeleri daha zayıf kaldı.

### H4: Mevsimsellik çok güçlüdür.
**Desteklendi.**  
Doluluk davranışında sezon etkisi açık biçimde görüldü.


## Bu Aşamada Ne Yapılmadı?

Bu notebook esas olarak **14 Nisan checkpoint** kapsamını karşılar.  
Bu yüzden burada yapılan analizler:

- veri toplama,
- veri temizleme,
- veri birleştirme,
- EDA,
- hipotez testi niteliğinde ilk istatistiksel incelemeler

ile sınırlıdır.

Henüz bu dosyanın ana odağı olmayan kısım:
- nihai makine öğrenmesi modeli seçimi,
- ileri model karşılaştırması,
- feature importance'ın son hali,
- final forecasting pipeline

değildir.

Bunlar bir sonraki proje aşamasına aittir.


## Sınırlılıklar

Bu aşamada bazı önemli sınırlılıklar da not edilmelidir:

### 1. Google Trends doğrudan rezervasyon verisi değildir
Bu veri sadece arama ilgisini temsil eder.

### 2. B2B yapı çok güçlü olabilir
Bu nedenle Trends etkisi doğal olarak sınırlı kalabilir.

### 3. Tüm ülkeler ve anahtar kelimeler aynı kalitede sinyal taşımıyor
Bazıları yararlı, bazıları gürültülü olabilir.

### 4. Korelasyon nedensellik değildir
Bir değişkenin dolulukla birlikte hareket etmesi, onun doğrudan sebep olduğu anlamına gelmez.

### 5. Mevsimsellik karıştırıcı rol oynayabilir
Hem Trends hem occupancy yazın artıyorsa, aradaki ilişki kısmen sezondan kaynaklanıyor olabilir.


## Bu Aşamadan Çıkan En Önemli Sonuç

Bu EDA çalışmasının en önemli mesajı şudur:

> Google Trends, resort otel doluluğunu aynı gün açıklayan çok güçlü bir değişken değil; fakat bazı gecikmeli sürümleri, doluluğu önceden haber veren yararlı sinyaller taşıyor olabilir.

Bu sonuç proje açısından değerlidir çünkü:
- başlangıçtaki iş hipotezini tamamen reddetmiyor,
- ama aynı zamanda Trends'i abartmıyor,
- daha dengeli ve gerçekçi bir yorum sunuyor.


## Sonraki Adım

Bu EDA aşamasından sonra mantıklı sonraki adım şudur:

1. en güçlü Trends özelliklerini seçmek,
2. bunların gecikmeli versiyonlarını oluşturmak,
3. takvim ve geçmiş doluluk bilgileriyle birlikte modelde kullanmak,
4. zaman bazlı train-test ayrımı ile gerçekçi performans ölçmek.

Yani bu notebook'un çıkardığı bilgi, doğrudan bir sonraki ML aşamasına girdi sağlar.


## Kısa Sonuç

Bu çalışma kapsamında:
- iki otelin günlük doluluk verisi hazırlandı,
- dört ülke için Google Trends verisi toplandı,
- tüm veriler tek bir master table içinde birleştirildi,
- veri kalitesi kontrolleri yapıldı,
- doluluk ve Trends değişkenleri keşifsel olarak incelendi,
- aynı gün ve gecikmeli ilişkiler test edildi,
- bazı gecikmeli Trends sinyallerinin umut verici olduğu görüldü.

Bu nedenle proje, 14 Nisan tesliminde beklenen:
- data collection,
- EDA,
- hypothesis tests

aşamalarını anlamlı ve savunulabilir biçimde karşılamaktadır.
